In [37]:
#Importing all the neccessary libraries
%matplotlib inline
import sqlite3
import pandas as pd
import numpy as np
import nltk
import string
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfTransformer, TfidfVectorizer, CountVectorizer
import sklearn.metrics as metrics
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_score, recall_score, f1_score
from nltk.stem.porter import PorterStemmer
import string
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
#from gensim.models import Word2Vec, KeyedVectors
import pickle
import warnings
warnings.filterwarnings("ignore")
from sklearn import datasets, neighbors
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from collections import Counter
from matplotlib.colors import ListedColormap
#import scikitplot.metrics as sciplot
from sklearn.metrics import accuracy_score
import math
import nltk
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\komal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [38]:
df_1=pd.read_csv(r"C:\Users\komal\Downloads\Reviews.csv")

In [39]:
df_1.head(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...


In [40]:
df_1.shape

(568454, 10)

In [41]:
df_1=df_1.sample(frac=0.1,random_state=1)

In [42]:
df_1.shape

(56845, 10)

In [43]:
df_1.head(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
288312,288313,B000ENUC3S,AN66F3Q4QNU43,Donna Speaker,0,0,5,1340496000,Cherry Pie Larabar,I love the Cherry Pie Lara bar. Best and tast...
431726,431727,B002TMV3CG,A3G007LQX6KGOD,SevereWX,0,0,5,1287878400,Melitta Coffee,Melitta Cafe COllection Blanc et Noir coffee h...


In [44]:
df_1['sentimentpolarity'] = df_1['Score'].apply(lambda x : 'Positive' if x>3 else 'Negative')

In [45]:
df_1.head(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,sentimentpolarity
288312,288313,B000ENUC3S,AN66F3Q4QNU43,Donna Speaker,0,0,5,1340496000,Cherry Pie Larabar,I love the Cherry Pie Lara bar. Best and tast...,Positive
431726,431727,B002TMV3CG,A3G007LQX6KGOD,SevereWX,0,0,5,1287878400,Melitta Coffee,Melitta Cafe COllection Blanc et Noir coffee h...,Positive


In [46]:
df_1['sentimentpolarity'].value_counts()

sentimentpolarity
Positive    44391
Negative    12454
Name: count, dtype: int64

In [47]:
df_1["class_label"] = df_1['sentimentpolarity'].apply(lambda x : 1 if x=="Positive" else 0)

In [48]:
df_1.head(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,sentimentpolarity,class_label
288312,288313,B000ENUC3S,AN66F3Q4QNU43,Donna Speaker,0,0,5,1340496000,Cherry Pie Larabar,I love the Cherry Pie Lara bar. Best and tast...,Positive,1
431726,431727,B002TMV3CG,A3G007LQX6KGOD,SevereWX,0,0,5,1287878400,Melitta Coffee,Melitta Cafe COllection Blanc et Noir coffee h...,Positive,1


In [49]:
df_1.shape

(56845, 12)

In [50]:
df_dup = df_1.drop_duplicates(subset={"UserId","ProfileName","Time","Text"},keep='first',inplace=True)

In [51]:
print(df_1["sentimentpolarity"].value_counts())

sentimentpolarity
Positive    40741
Negative    11450
Name: count, dtype: int64


In [52]:
final_data=df_1[df_1.HelpfulnessNumerator<= df_1.HelpfulnessDenominator]

In [53]:
final_data.head(2)

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,sentimentpolarity,class_label
288312,288313,B000ENUC3S,AN66F3Q4QNU43,Donna Speaker,0,0,5,1340496000,Cherry Pie Larabar,I love the Cherry Pie Lara bar. Best and tast...,Positive,1
431726,431727,B002TMV3CG,A3G007LQX6KGOD,SevereWX,0,0,5,1287878400,Melitta Coffee,Melitta Cafe COllection Blanc et Noir coffee h...,Positive,1


In [54]:
final_data["sentimentpolarity"].value_counts()

sentimentpolarity
Positive    40741
Negative    11450
Name: count, dtype: int64

In [55]:
sampled_dataset=final_data.drop(labels=['Id','ProductId', 'UserId', 'Score', 'ProfileName','HelpfulnessNumerator', 'HelpfulnessDenominator','Summary'], axis=1)
print("The shape of the sampled dataset after dropping unwanted columns : ", sampled_dataset.shape)
sampled_dataset.head()

The shape of the sampled dataset after dropping unwanted columns :  (52191, 4)


,Time,Text,sentimentpolarity,class_label
288312,1340496000,I love the Cherry Pie Lara bar. Best and tast...,Positive,1
431726,1287878400,Melitta Cafe COllection Blanc et Noir coffee h...,Positive,1
110311,1331769600,my girls absolutely loved this tuna. they were...,Positive,1
91855,1332806400,The vendor is fast and dependable. The tea is ...,Positive,1
338855,1271376000,UPDATE - 8/9/2010<br />A lot can happen in jus...,Positive,1


In [56]:
sampled_dataset=sampled_dataset.sort_values('Time', axis=0, ascending=False, inplace=False, kind='quicksort', na_position='last')

In [57]:
sampled_dataset = sampled_dataset.reset_index()

In [58]:
sampled_dataset.head(2)

,index,Time,Text,sentimentpolarity,class_label
0,317935,1351209600,It is hard to find much of anything sugarfree ...,Positive,1
1,271217,1351209600,Shake the container and they come running. Eve...,Positive,1


In [59]:
sampled_dataset=sampled_dataset.drop(labels=['index'], axis=1)

In [60]:
sampled_dataset["sentimentpolarity"].value_counts().plot(kind='bar',color=['green','red'],title='Distribution Of Positive and Negative reviews after De-Duplication.',figsize=(5,5))

<Axes: title={'center': 'Distribution Of Positive and Negative reviews after De-Duplication.'}, xlabel='sentimentpolarity'>

In [61]:
import re
review = "i am satisfied with the product , and the service is good... @  ! 7 % $ and the delivery is on time <br /> <br /> overall i am happy with the product.product"
def removeHtml(sentence): 
    pattern = re.compile('<.*?>')
    cleaned_text = re.sub(pattern,' ',sentence)
    return cleaned_text

In [62]:
Cleaned_review=removeHtml(review)
print ('the sentence :',Cleaned_review)

the sentence : i am satisfied with the product , and the service is good... @  ! 7 % $ and the delivery is on time     overall i am happy with the product.product


In [64]:
def removePunctuations(sentence):
    cleaned_text  = re.sub('[^a-zA-Z]',' ',sentence)
    return cleaned_text
cleaned_review_only_text=removePunctuations(Cleaned_review)
print('the sentence after remove punctuation :',cleaned_review_only_text)

the sentence after remove punctuation : i am satisfied with the product   and the service is good               and the delivery is on time     overall i am happy with the product product


In [65]:
def stripLower(sentence):
    cleaned_text = sentence.lower().strip()
    cleaned_text = re.sub(' +', ' ', cleaned_text)
    return cleaned_text
final_cleaned_review = stripLower(cleaned_review_only_text)
print("The review after converting to lower case and stripping extra spaces : ",final_cleaned_review)

The review after converting to lower case and stripping extra spaces :  i am satisfied with the product and the service is good and the delivery is on time overall i am happy with the product product


In [66]:
def take_only_distinct_words(sentence):
    word_tokens = sentence.split()
    #print("The word tokens are : ", word_tokens)
    seen = set()
    cleaned_text = []
    for word in word_tokens:
        if word not in seen:
            cleaned_text.append(word)
            seen.add(word)
    #print("The distinct words are : ", cleaned_text)
    cleaned_text = ' '.join(cleaned_text)
    return cleaned_text

cleaned_review_distinct_words = take_only_distinct_words(final_cleaned_review)
print("The review after taking only distinct words : ", cleaned_review_distinct_words)

The review after taking only distinct words :  i am satisfied with the product and service is good delivery on time overall happy


In [67]:
import nltk   #spacy
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\komal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [68]:
nltk.download('stopwords')
def removeStopWords(sentence):
    stop_words = set(stopwords.words('english'))
    # need to take all negative words from the stop words in list after that we will remove those words from the stop words
    #print("The stop words are : ", stop_words)
    negative_words = ["not", "no", "nor", "never", "don't", "didn't", "doesn't", "isn't", "wasn't", "shouldn't", "wouldn't", "couldn't", "won't", "haven't", "hasn't", "hadn't", "mightn't", "mustn't", "shan't"]
    #print("The negative words are : ", negative_words)
    stop_words = stop_words - set(negative_words)
    word_tokens = sentence.split()
    cleaned_text = [word for word in word_tokens if not word in stop_words]
    cleaned_text = ' '.join(cleaned_text)
    return cleaned_text

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\komal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [71]:
rev_after_stopwords = removeStopWords(cleaned_review_distinct_words)

In [72]:
rev_test = "satisfy  satisfied  satisfied satisfaction satisfactory satisfyingly"
# next step is to use stemming or  lemmatisation on the text data:
from nltk.stem.porter import PorterStemmer
def stemming(sentence):
    porter_stemmer = PorterStemmer()
    word_tokens = sentence.split()
    cleaned_text = [porter_stemmer.stem(word) for word in word_tokens]
    cleaned_text = ' '.join(cleaned_text)
    return cleaned_text
review_after_stemming = stemming(rev_test)
print("The review after stemming : ",review_after_stemming)

The review after stemming :  satisfi satisfi satisfi satisfact satisfactori satisfyingli


In [73]:
# create a function to use the stemming technique with snowball stemmer
def stemming_with_snowball(sentence):
    from nltk.stem.snowball import SnowballStemmer
    snowball_stemmer = SnowballStemmer(language='english')
    word_tokens = sentence.split()
    cleaned_text = [snowball_stemmer.stem(word) for word in word_tokens]
    cleaned_text = ' '.join(cleaned_text)
    return cleaned_text
review_after_snowball_stemming = stemming_with_snowball(rev_test)
print("The review after snowball stemmer : ",review_after_snowball_stemming)

The review after snowball stemmer :  satisfi satisfi satisfi satisfact satisfactori satisfi


In [74]:
def lemmatization(sentence):
    lemmatizer = WordNetLemmatizer()
    word_tokens = sentence.split()
    cleaned_text = [lemmatizer.lemmatize(word) for word in word_tokens]
    cleaned_text = ' '.join(cleaned_text)
    return cleaned_text

review_after_lemmatization = lemmatization(rev_test)
print("The review after lemmatization : ",review_after_lemmatization)

The review after lemmatization :  satisfy satisfied satisfied satisfaction satisfactory satisfyingly


In [75]:
sampled_dataset.head(2)

,Time,Text,sentimentpolarity,class_label
0,1351209600,It is hard to find much of anything sugarfree ...,Positive,1
1,1351209600,Shake the container and they come running. Eve...,Positive,1


In [77]:
def data_preprocessing(sentence):
    sentence = removeHtml(sentence)
    sentence = removePunctuations(sentence)
    sentence = stripLower(sentence)
    sentence = take_only_distinct_words(sentence)
    sentence = removeStopWords(sentence)
    sentence = stemming(sentence)
    return sentence

sampled_dataset['Cleaned_Text'] = sampled_dataset['Text'].apply(data_preprocessing)

In [78]:
sampled_dataset.head(2)

,Time,Text,sentimentpolarity,class_label,Cleaned_Text
0,1351209600,It is hard to find much of anything sugarfree ...,Positive,1,hard find much anyth sugarfre realli tast good...
1,1351209600,Shake the container and they come running. Eve...,Positive,1,shake contain come run even boy cat big food l...


In [80]:
sampled_dataset = sampled_dataset[['Time','Cleaned_Text','class_label']]

In [81]:
sampled_dataset.shape

(52191, 3)

In [82]:
sampled_dataset.head(2)

,Time,Cleaned_Text,class_label
0,1351209600,hard find much anyth sugarfre realli tast good...,1
1,1351209600,shake contain come run even boy cat big food l...,1


In [83]:
sampled_dataset['class_label'].value_counts()

class_label
1    40741
0    11450
Name: count, dtype: int64

In [88]:
def splitting_data(data):
    X = data['Cleaned_Text']
    y = data['class_label']
    return X,y

In [89]:
X,y = splitting_data(sampled_dataset)

In [90]:
X

0        hard find much anyth sugarfre realli tast good...
1        shake contain come run even boy cat big food l...
2        like spong candi never tri realli miss not abl...
3        everi month give three dog two aussi golden fl...
4        dog love healthi treat not great train crumbl ...
                               ...                        
52186    beetlejuic not movi watch time one funiest mov...
52187    set small new england town tim burton masterpi...
52188    michael keaton alreadi way major star play gho...
52189    beetlejuic awe inspir wonder amus comed romp e...
52190    twist rumplestiskin captur film star michael k...
Name: Cleaned_Text, Length: 52191, dtype: object

In [91]:
y

0        1
1        1
2        1
3        1
4        1
        ..
52186    1
52187    1
52188    1
52189    1
52190    1
Name: class_label, Length: 52191, dtype: int64

In [92]:
split = math.floor(0.8*len(X))
print("The split value is : ", split)
X_train = X[0:split,] ; y_train = y[0:split,]

X_test = X[split:,] ; y_test = y[split:,]

The split value is :  41752


In [93]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(41752,)
(10439,)
(41752,)
(10439,)


In [94]:
X_train

0        hard find much anyth sugarfre realli tast good...
1        shake contain come run even boy cat big food l...
2        like spong candi never tri realli miss not abl...
3        everi month give three dog two aussi golden fl...
4        dog love healthi treat not great train crumbl ...
                               ...                        
41747    notic quit peopl seem believ label product mad...
41748    avid tea drinker drink lot green everi day gre...
41749               order liter ate day pack addict delici
41750    displeas amazon descript say raw realiti label...
41751    pro gevalia say dark chocol mean con sinc oz b...
Name: Cleaned_Text, Length: 41752, dtype: object

In [95]:
X_test

41752    good earth restaur tea addict continu onlin pu...
41753    buy chai world market stop sell tri mani brand...
41754    nice light simpl yet flavor dress receiv compl...
41755    son not eat wheat dairi make food gf df great ...
41756    bj brewhous start serv douw egbert coffe year ...
                               ...                        
52186    beetlejuic not movi watch time one funiest mov...
52187    set small new england town tim burton masterpi...
52188    michael keaton alreadi way major star play gho...
52189    beetlejuic awe inspir wonder amus comed romp e...
52190    twist rumplestiskin captur film star michael k...
Name: Cleaned_Text, Length: 10439, dtype: object

In [96]:
y_train

0        1
1        1
2        1
3        1
4        1
        ..
41747    1
41748    1
41749    1
41750    0
41751    1
Name: class_label, Length: 41752, dtype: int64

In [97]:
y_test

41752    1
41753    1
41754    1
41755    1
41756    1
        ..
52186    1
52187    1
52188    1
52189    1
52190    1
Name: class_label, Length: 10439, dtype: int64

In [98]:
def total_unique_words(corpus):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(corpus)
    unique_words = vectorizer.get_feature_names_out()
    return unique_words, len(unique_words)

unique_words, num_unique_words = total_unique_words(X_train)
print("Total unique words:", num_unique_words)
print("Sample unique words:", unique_words[:20])


Total unique words: 25399
Sample unique words: ['aa' 'aaa' 'aaaa' 'aaaaa' 'aaaaaaaaaaaaaaaaaaaargh' 'aaaaaah' 'aaaaallll'
 'aaaahhhhhh' 'aaah' 'aabsolut' 'aacut' 'aafco' 'aah' 'aakaufman'
 'aalmost' 'aamazon' 'aap' 'aargh' 'aaround' 'aarti']


In [133]:
import pickle
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)


In [99]:
def text_to_bow(corpus):
    cv_object = CountVectorizer()
    X_bow = cv_object.fit_transform(corpus)
    # we need to save this cv_object in pickle file for test data conversion
    # saving the count vectorizer object in pickle file
    with open('count_vectorizer.pkl', 'wb') as f:
        pickle.dump(cv_object, f)
    return X_bow, cv_object


X_bow , cv_object = text_to_bow(X_train)

In [100]:
X_bow_array = X_bow.toarray()

In [101]:
X_bow.shape

(41752, 25399)

In [102]:
X_train

0        hard find much anyth sugarfre realli tast good...
1        shake contain come run even boy cat big food l...
2        like spong candi never tri realli miss not abl...
3        everi month give three dog two aussi golden fl...
4        dog love healthi treat not great train crumbl ...
                               ...                        
41747    notic quit peopl seem believ label product mad...
41748    avid tea drinker drink lot green everi day gre...
41749               order liter ate day pack addict delici
41750    displeas amazon descript say raw realiti label...
41751    pro gevalia say dark chocol mean con sinc oz b...
Name: Cleaned_Text, Length: 41752, dtype: object

In [103]:
model = MultinomialNB()

In [104]:
model.fit(X_bow,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [105]:
y_train_pred = model.predict(X_bow)

In [106]:
print("Accuracy:",metrics.accuracy_score(y_train, y_train_pred))

Accuracy: 0.888196972600115


In [107]:
# confusion matrix
cm = confusion_matrix(y_train, y_train_pred)
print("Confusion Matrix:\n", cm)

Confusion Matrix:
 [[ 6643  2995]
 [ 1673 30441]]


In [108]:
from sklearn.metrics import classification_report
print("Classification Report on train data:\n", classification_report(y_train, y_train_pred))

Classification Report on train data:
               precision    recall  f1-score   support

           0       0.80      0.69      0.74      9638
           1       0.91      0.95      0.93     32114

    accuracy                           0.89     41752
   macro avg       0.85      0.82      0.83     41752
weighted avg       0.88      0.89      0.89     41752



In [109]:
# we need to convert the text data into numerical data by using saved count vectorizer object
# loading the count vectorizer object from pickle file
#Create a function to convert text data into numerical data using Bag of words for test data conversion
def text_to_bow_test(corpus):
    with open('count_vectorizer.pkl', 'rb') as f:
        loaded_cv_object = pickle.load(f)
    X_bow = loaded_cv_object.transform(corpus)
    return X_bow

X_test_bow = text_to_bow_test(X_test)

In [110]:
y_test_pred = model.predict(X_test_bow)

In [111]:
print("Classification Report on test data:\n", classification_report(y_test, y_test_pred))

Classification Report on test data:
               precision    recall  f1-score   support

           0       0.68      0.52      0.59      1812
           1       0.90      0.95      0.93      8627

    accuracy                           0.88     10439
   macro avg       0.79      0.74      0.76     10439
weighted avg       0.87      0.88      0.87     10439



In [112]:
from sklearn.tree import DecisionTreeClassifier
dt_model = DecisionTreeClassifier()
dt_model.fit(X_bow, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [113]:
y_train_pred_dt = dt_model.predict(X_bow)
y_test_pred_dt = dt_model.predict(X_test_bow)
print("Classification Report on train data using Decision Tree Classifier:\n", classification_report(y_train,y_train_pred_dt))
print("Classification Report on test data using Decision Tree Classifier:\n", classification_report(y_test,y_test_pred_dt))

Classification Report on train data using Decision Tree Classifier:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      9638
           1       1.00      1.00      1.00     32114

    accuracy                           1.00     41752
   macro avg       1.00      1.00      1.00     41752
weighted avg       1.00      1.00      1.00     41752

Classification Report on test data using Decision Tree Classifier:
               precision    recall  f1-score   support

           0       0.43      0.45      0.44      1812
           1       0.88      0.88      0.88      8627

    accuracy                           0.80     10439
   macro avg       0.66      0.66      0.66     10439
weighted avg       0.81      0.80      0.80     10439



In [115]:
 #now we will try on Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier()
rf_model.fit(X_bow, y_train)
y_train_pred_rf = rf_model.predict(X_bow)
y_test_pred_rf = rf_model.predict(X_test_bow)   
print("Classification Report on train data using Random Forest Classifier:\n", classification_report(y_train,y_train_pred_rf))
print("Classification Report on test data using Random Forest Classifier:\n", classification_report(y_test,y_test_pred_rf))

Classification Report on train data using Random Forest Classifier:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      9638
           1       1.00      1.00      1.00     32114

    accuracy                           1.00     41752
   macro avg       1.00      1.00      1.00     41752
weighted avg       1.00      1.00      1.00     41752

Classification Report on test data using Random Forest Classifier:
               precision    recall  f1-score   support

           0       0.90      0.23      0.36      1812
           1       0.86      0.99      0.92      8627

    accuracy                           0.86     10439
   macro avg       0.88      0.61      0.64     10439
weighted avg       0.87      0.86      0.83     10439



In [117]:
pip install imbalanced_learn

Note: you may need to restart the kernel to use updated packages.


In [129]:
"""class label o having 10k rows 
and class label1 having 30k rows
The data set is imbalanced data set
# we need to do balancing the data set using SMOTE or ADASYN or RandomOverSampler or RandomUnderSampler"""
from imblearn.over_sampling import SMOTE
smote = SMOTE()
X_resampled, y_resampled = smote.fit_resample(X, y)

ValueError: could not convert string to float: 'hard find much anyth sugarfre realli tast good appl cider best also love regular hot form'

In [121]:
pip install xgboost

   ---------------------------------------- 0.0/56.8 MB ? eta -:--:--
   ---- ----------------------------------- 5.8/56.8 MB 32.2 MB/s eta 0:00:02
   ------- -------------------------------- 10.7/56.8 MB 28.0 MB/s eta 0:00:02
   -------- ------------------------------- 12.6/56.8 MB 21.3 MB/s eta 0:00:03
   --------- ------------------------------ 13.6/56.8 MB 17.1 MB/s eta 0:00:03
   ---------- ----------------------------- 14.9/56.8 MB 14.9 MB/s eta 0:00:03
   ----------- ---------------------------- 15.7/56.8 MB 12.7 MB/s eta 0:00:04
   ----------- ---------------------------- 17.0/56.8 MB 11.8 MB/s eta 0:00:04
   ------------- -------------------------- 18.9/56.8 MB 11.2 MB/s eta 0:00:04
   -------------- ------------------------- 20.7/56.8 MB 11.1 MB/s eta 0:00:04
   ---------------- ----------------------- 23.1/56.8 MB 11.1 MB/s eta 0:00:04
   ------------------ --------------------- 25.7/56.8 MB 11.1 MB/s eta 0:00:03
   -------------------- ------------------- 28.6/56.8 MB 11.3 

In [122]:
import xgboost as xgb
xgb_model = xgb.XGBClassifier()
xgb_model.fit(X_bow, y_train)
y_train_pred_xgb = xgb_model.predict(X_bow)
y_test_pred_xgb = xgb_model.predict(X_test_bow)
print("Classification Report on train data using XGBoost Classifier:\n", classification_report(y_train,y_train_pred_xgb))
print("Classification Report on test data using XGBoost Classifier:\n", classification_report(y_test,y_test_pred_xgb))

Classification Report on train data using XGBoost Classifier:
               precision    recall  f1-score   support

           0       0.89      0.65      0.75      9638
           1       0.90      0.98      0.94     32114

    accuracy                           0.90     41752
   macro avg       0.90      0.82      0.85     41752
weighted avg       0.90      0.90      0.90     41752

Classification Report on test data using XGBoost Classifier:
               precision    recall  f1-score   support

           0       0.72      0.46      0.56      1812
           1       0.89      0.96      0.93      8627

    accuracy                           0.87     10439
   macro avg       0.81      0.71      0.74     10439
weighted avg       0.86      0.87      0.86     10439



In [123]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2'],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced'],
    'criterion': ['gini', 'entropy']
}


In [124]:
rf_model_tuned = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5, min_samples_leaf=1, max_features='sqrt', bootstrap=True, class_weight=None, criterion='gini')
rf_model_tuned.fit(X_bow, y_train)
y_train_pred_rf_tuned = rf_model_tuned.predict(X_bow)
y_test_pred_rf_tuned = rf_model_tuned.predict(X_test_bow)
print("Classification Report on train data using Tuned Random Forest Classifier:\n", classification_report(y_train,y_train_pred_rf_tuned))
print("Classification Report on test data using Tuned Random Forest Classifier:\n", classification_report(y_test,y_test_pred_rf_tuned))

Classification Report on train data using Tuned Random Forest Classifier:
               precision    recall  f1-score   support

           0       1.00      0.03      0.05      9638
           1       0.77      1.00      0.87     32114

    accuracy                           0.78     41752
   macro avg       0.89      0.51      0.46     41752
weighted avg       0.83      0.78      0.68     41752

Classification Report on test data using Tuned Random Forest Classifier:
               precision    recall  f1-score   support

           0       1.00      0.00      0.00      1812
           1       0.83      1.00      0.91      8627

    accuracy                           0.83     10439
   macro avg       0.91      0.50      0.45     10439
weighted avg       0.86      0.83      0.75     10439



In [125]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_bow, y_train)

In [126]:
X_resampled.shape

(64228, 25399)

In [127]:
y_resampled

0        1
1        1
2        1
3        1
4        1
        ..
64223    0
64224    0
64225    0
64226    0
64227    0
Name: class_label, Length: 64228, dtype: int64

In [128]:
# build the random forest model on the resampled data
rf_model_tuned = RandomForestClassifier(n_estimators=200, max_depth=20, min_samples_split=5, min_samples_leaf=1, max_features='sqrt', bootstrap=True, class_weight=None, criterion='gini')
rf_model_tuned.fit(X_resampled, y_resampled)
y_train_pred_rf_tuned = rf_model_tuned.predict(X_bow)
y_test_pred_rf_tuned = rf_model_tuned.predict(X_test_bow)
print("Classification Report on train data using Tuned Random Forest Classifier after OverSampling:\n", classification_report(y_train,y_train_pred_rf_tuned))
print("Classification Report on test data using Tuned Random Forest Classifier after OverSampling:\n", classification_report(y_test,y_test_pred_rf_tuned))

Classification Report on train data using Tuned Random Forest Classifier after OverSampling:
               precision    recall  f1-score   support

           0       0.74      0.82      0.78      9638
           1       0.95      0.91      0.93     32114

    accuracy                           0.89     41752
   macro avg       0.84      0.87      0.85     41752
weighted avg       0.90      0.89      0.89     41752

Classification Report on test data using Tuned Random Forest Classifier after OverSampling:
               precision    recall  f1-score   support

           0       0.57      0.66      0.61      1812
           1       0.93      0.89      0.91      8627

    accuracy                           0.85     10439
   macro avg       0.75      0.78      0.76     10439
weighted avg       0.86      0.85      0.86     10439



In [130]:
pip install streamlit

   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   -------------------- ------------------- 5.2/10.1 MB 29.0 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 29.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/731.2 kB ? eta -:--:--
   --------------------------------------- 731.2/731.2 kB 15.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.9 MB ? eta -:--:--
   ---------------- ----------------------- 2.9/6.9 MB 14.0 MB/s eta 0:00:01
   ---------------------- ----------------- 3.9/6.9 MB 9.8 MB/s eta 0:00:01
   ---------------------------- ----------- 5.0/6.9 MB 8.2 MB/s eta 0:00:01
   ------------------------------------ --- 6.3/6.9 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 6.9/6.9 MB 7.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/26.2 MB ? eta -:--:--
   -- ------------------------------------- 1.6/26.2 MB 8.4 MB/s eta 0:00:03
   ----- ----------------

In [134]:
import streamlit as st
import pickle

# Load the saved model and vectorizer
with open('model.pkl', 'rb') as model_file:
    model = pickle.load(model_file)

with open('count_vectorizer.pkl', 'rb') as vec_file:
    vectorizer = pickle.load(vec_file)

# Title
st.title("Review Sentiment Classifier")

# Input box
user_input = st.text_area("Enter your review:")

if st.button("Predict"):
    if user_input.strip() == "":
        st.warning("Please enter some text.")
    else:
        # Transform input text
        input_vector = vectorizer.transform([user_input])
        
        # Make prediction
        prediction = model.predict(input_vector)[0]
        
        # Show result
        if prediction == 1:
            st.success("Prediction: Positive ✅")
        else:
            st.error("Prediction: Negative ❌")

2025-10-03 12:07:44.537 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-03 12:07:44.842 
  command:

    streamlit run C:\Users\komal\.conda\envs\myenv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2025-10-03 12:07:44.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-03 12:07:44.844 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-03 12:07:44.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-03 12:07:44.846 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-03 12:07:44.848 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-03 12:07:44.849 Thread 'MainThr